# Stance Detection Transformer Review

This notebook reviews the stance DistilBERT model and compares it with the classical and deep-learning models when those result files exist.

Training lives in `script/train_stance_transformer.py`. Benchmark evaluation lives in `script/run_benchmarks.py`.

## Run First

Run these scripts from PyCharm or from the project root:

```bash
uv run python script/train_stance_transformer.py
uv run python script/run_benchmarks.py
```

Benchmark outputs are saved under `artifacts/evaluation/stance/fnc1/<timestamp>/`.

In [3]:
import json
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(r"C:\Users\Barderus_Legion\PycharmProjects\TrustNet")
TRANSFORMER_RESULTS_FOLDER = PROJECT_ROOT / "artifacts" / "evaluation" / "stance" / "fnc1"

transformer_runs = sorted([folder for folder in TRANSFORMER_RESULTS_FOLDER.glob("*") if folder.is_dir()], reverse=True) if TRANSFORMER_RESULTS_FOLDER.exists() else []
latest_transformer_run = transformer_runs[0] if transformer_runs else None
latest_transformer_run

## Transformer Benchmark Metrics

This table shows the most recent benchmark metrics for the stance transformer model.

In [4]:
if latest_transformer_run is None:
    print("No stance transformer benchmark found yet. Run script/run_benchmarks.py after training the model.")
else:
    with (latest_transformer_run / "metrics.json").open("r", encoding="utf-8") as file_handle:
        metrics = json.load(file_handle)
    transformer_table = pd.DataFrame([
        {
            "model_type": "transformer",
            "model": "distilbert_stance",
            "accuracy": metrics.get("accuracy"),
            "macro_precision": metrics.get("macro_precision"),
            "macro_recall": metrics.get("macro_recall"),
            "macro_f1": metrics.get("macro_f1"),
            "weighted_f1": metrics.get("weighted_f1"),
            "roc_auc": metrics.get("roc_auc"),
            "log_loss": metrics.get("log_loss"),
        }
    ])
    display(transformer_table)

No stance transformer benchmark found yet. Run script/run_benchmarks.py after training the model.


## All Stance Model Comparison

This table combines the latest available classical, deep-learning, and transformer results. If a row is missing, run that model's script first.

In [5]:
comparison_rows = []

baseline_folder = PROJECT_ROOT / "artifacts" / "baselines" / "stance"
baseline_runs = sorted([folder for folder in baseline_folder.glob("*") if folder.is_dir()], reverse=True) if baseline_folder.exists() else []
if baseline_runs:
    baseline_metrics = pd.read_csv(baseline_runs[0] / "metrics.csv")
    baseline_metrics["model_type"] = "classical"
    comparison_rows.append(baseline_metrics)

deep_learning_file = PROJECT_ROOT / "artifacts" / "deep_learning" / "stance" / "metrics.csv"
if deep_learning_file.exists():
    deep_learning_metrics = pd.read_csv(deep_learning_file)
    deep_learning_metrics["model_type"] = "deep_learning"
    comparison_rows.append(deep_learning_metrics)

if latest_transformer_run is not None:
    with (latest_transformer_run / "metrics.json").open("r", encoding="utf-8") as file_handle:
        metrics = json.load(file_handle)
    comparison_rows.append(pd.DataFrame([{
        "model_type": "transformer",
        "model": "distilbert_stance",
        "accuracy": metrics.get("accuracy"),
        "macro_precision": metrics.get("macro_precision"),
        "macro_recall": metrics.get("macro_recall"),
        "macro_f1": metrics.get("macro_f1"),
        "weighted_f1": metrics.get("weighted_f1"),
        "roc_auc": metrics.get("roc_auc"),
    }]))

if not comparison_rows:
    print("No model results found yet.")
else:
    all_results = pd.concat(comparison_rows, ignore_index=True, sort=False)
    columns = ["model_type", "model", "accuracy", "macro_precision", "macro_recall", "macro_f1", "weighted_f1", "roc_auc"]
    all_results = all_results[[column for column in columns if column in all_results.columns]]
    all_results = all_results.sort_values("macro_f1", ascending=False)
    display(all_results)

,model_type,model,accuracy,macro_precision,macro_recall,macro_f1,weighted_f1,roc_auc
4,deep_learning,bidirectional_lstm,0.876438,0.703304,0.656036,0.677379,0.873245,0.953611
1,classical,linear_svc_tfidf,0.837119,0.708071,0.514014,0.575180,0.816498,NaN
3,deep_learning,text_cnn,0.831516,0.669123,0.522751,0.561758,0.813148,0.903173
2,classical,ridge_classifier_tfidf,0.825813,0.691004,0.483698,0.543228,0.801166,NaN
0,classical,logistic_regression_tfidf,0.817709,0.705142,0.450466,0.509591,0.788012,NaN


## Discussion Notes

Use this comparison to discuss which model family performed best for stance detection. Macro F1 should be emphasized because the stance labels are imbalanced and minority-class mistakes matter for the research question.